### Importy i Funkcje Pomocnicze

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, window, collect_set, count
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
import os # Dodano import os

# --- Konfiguracja globalna i funkcje pomocnicze ---
_batch_counter_global = {"count": 0}
spark_session_global = None # Placeholder dla globalnej sesji Spark

def process_batch_display(df, batch_id):
    """
    Pomocnicza funkcja do wyświetlania danych strumieniowych.
    Nie zatrzymuje automatycznie strumienia ani sesji Spark.
    Licznik batchy jest globalny dla wszystkich strumieni używających tej funkcji.
    """
    global _batch_counter_global
    _batch_counter_global["count"] += 1
    print(f"--- Batch ID: {batch_id} (Global display count: {_batch_counter_global['count']}) ---")
    df.show(truncate=False)

def reset_global_batch_counter():
    """Resetuje globalny licznik batchy dla process_batch_display."""
    global _batch_counter_global
    _batch_counter_global["count"] = 0
    print("Global batch display counter has been reset.")

def create_spark_session(appName="StreamingAppJupyter"):
    """
    Tworzy lub pobiera istniejącą sesję Spark i przypisuje ją do spark_session_global.
    Automatycznie konfiguruje 'hadoop.home.dir'.
    """
    global spark_session_global

    hadoop_home_path = os.environ.get('HADOOP_HOME', 'C:\\hadoop')

    if spark_session_global is None or spark_session_global._sc.isStopped:
        spark_session_global = (SparkSession.builder
                                .appName(appName)
                                .config("spark.sql.shuffle.partitions", "2") 
                                .config("spark.sql.streaming.stopActiveRunOnRestart", "true")
                                .config("hadoop.home.dir", hadoop_home_path) # <-- KLUCZOWA KONFIGURACJA
                                .getOrCreate())
        spark_session_global.sparkContext.setLogLevel("ERROR")
    return spark_session_global

# Schemat danych dla plików JSON (używany w wielu zadaniach)
json_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True)
])

# Ścieżka do danych generowanych przez generator.py
input_files_path = "data/stream"

### Generator danych

In [ ]:
%%file generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream" # Upewnij się, że ta ścieżka zgadza się z input_files_path
os.makedirs(output_dir, exist_ok=True)

event_types = ["view", "cart", "purchase"]
categories = ["electronics", "books", "fashion", "home", "sports"]

def generate_event():
    return {
        "user_id": f"u{random.randint(1, 10)}",
        "event_type": random.choices(event_types, weights=[0.6, 0.25, 0.15])[0],
        "timestamp": (datetime.utcnow() - timedelta(seconds=random.randint(0, 300))).isoformat() + "Z",
        "product_id": f"p{random.randint(100, 120)}",
        "category": random.choice(categories),
        "price": round(random.uniform(10, 500), 2)
    }

try:
    file_counter = 0
    while True:
        batch = [generate_event() for _ in range(random.randint(5, 15))]
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{output_dir}/events_{timestamp_str}_{file_counter}.json"
        with open(filename, "w") as f:
            for e in batch:
                f.write(json.dumps(e) + "\n")
        # print(f"Wrote: {filename} ({len(batch)} events)") # Można odkomentować dla szczegółowych logów
        file_counter += 1
        time.sleep(random.uniform(4, 8)) # Generuj pliki co 4-8 sekund
except KeyboardInterrupt:
    print("\nData generator stopped by user.")

### Uruchomienie w terminalu generatora (python generator.py)
### Inicjalizacja sesji Spark

In [2]:
spark = create_spark_session("RTA_sesja")
# spark_session_global jest teraz ustawiony i dostępny globalnie

### Zadanie 1 - rate jako źródło kontrolowanego strumienia

In [3]:
print("\n--- URUCHAMIAM ZADANIE 1: Rate Source ---")
reset_global_batch_counter() # Resetuj licznik dla tego zadania

rate_df_task1 = (spark.readStream  # Używamy globalnej sesji 'spark'
           .format("rate")
           .option("rowsPerSecond", 3) # Zmniejszono dla lepszej obserwacji
           .load())

events_task1 = (rate_df_task1
          .withColumn("user_id", expr("concat('u', cast(rand()*100 as int))"))
          .withColumn("event_type", expr("case when rand() > 0.7 then 'purchase' else 'view' end")))
    
query_task1 = (events_task1.writeStream
         .format("console")
         .outputMode("update") 
         .option("truncate", False) # Dodano dla pełnego widoku w konsoli
         .foreachBatch(process_batch_display) # Używamy zmodyfikowanej funkcji
         .queryName("RateSourceQuery")
         .start())

print("Rate source query (Task 1) started. Processing for a short duration...")
try:
    query_task1.awaitTermination(timeout=15)
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_task1.isActive:
    print("Stopping RateSourceQuery (Task 1)...")
    query_task1.stop()
    # Czekaj aż strumień faktycznie się zatrzyma
    for _ in range(10): # max 5 sekund czekania
        if not query_task1.isActive: break
        time.sleep(0.5)
    if query_task1.isActive: print("Query did not stop gracefully.")
    
print("Rate source query (Task 1) finished or stopped.")


--- URUCHAMIAM ZADANIE 1: Rate Source ---
Global batch display counter has been reset.
Rate source query (Task 1) started. Processing for a short duration...
--- Batch ID: 0 (Global display count: 1) ---
+---------+-----+-------+----------+
|timestamp|value|user_id|event_type|
+---------+-----+-------+----------+
+---------+-----+-------+----------+

--- Batch ID: 1 (Global display count: 2) ---
+-----------------------+-----+-------+----------+
|timestamp              |value|user_id|event_type|
+-----------------------+-----+-------+----------+
|2025-05-17 17:02:59.893|0    |u18    |purchase  |
|2025-05-17 17:03:00.226|1    |u8     |view      |
|2025-05-17 17:03:00.56 |2    |u90    |view      |
|2025-05-17 17:03:00.893|3    |u93    |view      |
|2025-05-17 17:03:01.226|4    |u97    |view      |
|2025-05-17 17:03:01.56 |5    |u37    |view      |
+-----------------------+-----+-------+----------+

--- Batch ID: 2 (Global display count: 3) ---
+-----------------------+-----+-------+----

### Zadanie 2 - Filtrowanie danych bez agregacji (append mode)

In [3]:
print("\n--- URUCHAMIAM ZADANIE 2: Filtering (Append Mode) ---")
reset_global_batch_counter()

# Źródło rate
rate_df_task2 = (spark.readStream
           .format("rate")
           .option("rowsPerSecond", 3)
           .load())
events_task2 = (rate_df_task2
          .withColumn("user_id", expr("concat('u', cast(rand()*100 as int))")) # Ponownie generujemy, niezależnie od zad 1
          .withColumn("event_type", expr("case when rand() > 0.7 then 'purchase' else 'view' end")))

# Wyfiltruj tylko purchase
purchases_task2 = events_task2.filter(col("event_type") == "purchase")

query_task2 = (purchases_task2.writeStream
         .format("console")
         .outputMode("append") 
         .option("truncate", False)
         .foreachBatch(process_batch_display)
         .queryName("FilteringQuery")
         .start())

print("Filtering query (Task 2) started. Processing for a short duration...")
try:
    query_task2.awaitTermination(timeout=15) 
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_task2.isActive:
    print("Stopping FilteringQuery (Task 2)...")
    query_task2.stop()
    for _ in range(10): 
        if not query_task2.isActive: break
        time.sleep(0.5)
print("Filtering query (Task 2) finished or stopped.")


--- URUCHAMIAM ZADANIE 2: Filtering (Append Mode) ---
Global batch display counter has been reset.
Filtering query (Task 2) started. Processing for a short duration...
--- Batch ID: 0 (Global display count: 1) ---
+---------+-----+-------+----------+
|timestamp|value|user_id|event_type|
+---------+-----+-------+----------+
+---------+-----+-------+----------+

--- Batch ID: 1 (Global display count: 2) ---
+-----------------------+-----+-------+----------+
|timestamp              |value|user_id|event_type|
+-----------------------+-----+-------+----------+
|2025-05-17 17:04:46.383|4    |u86    |purchase  |
|2025-05-17 17:04:46.717|5    |u28    |purchase  |
+-----------------------+-----+-------+----------+

--- Batch ID: 2 (Global display count: 3) ---
+-----------------------+-----+-------+----------+
|timestamp              |value|user_id|event_type|
+-----------------------+-----+-------+----------+
|2025-05-17 17:04:47.05 |6    |u75    |purchase  |
|2025-05-17 17:04:47.383|7    |u3

### Zadanie 3 - Źródło plikowe (JSON) - Podstawowy odczyt

In [3]:
print("\n--- URUCHAMIAM ZADANIE 3: File Source (JSON) ---")
reset_global_batch_counter()

# input_files_path i json_schema są zdefiniowane globalnie w Komórce 1

stream_task3 = (spark.readStream
          .schema(json_schema) # Używamy globalnego schematu
          .option("maxFilesPerTrigger", 1) # Przetwarzaj jeden nowy plik na batch
          .json(input_files_path)) # Używamy globalnej ścieżki

query_task3 = (stream_task3.writeStream
         .format("console")
         .outputMode("append")
         .option("truncate", False)
         .foreachBatch(process_batch_display)
         .queryName("JSONFileQuery")
         .start())

print(f"JSON file source query (Task 3) started. Monitoring: {input_files_path}")
print("Ensure generator.py is running. Processing for a short duration...")
try:
    query_task3.awaitTermination(timeout=15) 
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_task3.isActive:
    print("Stopping JSONFileQuery (Task 3)...")
    query_task3.stop()
    for _ in range(10): 
        if not query_task3.isActive: break
        time.sleep(0.5)
print("JSON file source query (Task 3) finished or stopped.")


--- URUCHAMIAM ZADANIE 3: File Source (JSON) ---
Global batch display counter has been reset.
JSON file source query (Task 3) started. Monitoring: data/stream
Ensure generator.py is running. Processing for a short duration...
--- Batch ID: 0 (Global display count: 1) ---
+-------+----------+--------------------------+----------+-----------+------+
|user_id|event_type|timestamp                 |product_id|category   |price |
+-------+----------+--------------------------+----------+-----------+------+
|u6     |view      |2025-05-17 16:00:33.960275|p100      |sports     |485.05|
|u7     |view      |2025-05-17 16:00:38.960314|p107      |home       |123.39|
|u10    |view      |2025-05-17 16:00:00.960333|p102      |home       |83.8  |
|u5     |view      |2025-05-17 16:01:30.960344|p103      |books      |144.02|
|u8     |view      |2025-05-17 16:02:09.960353|p107      |home       |494.4 |
|u4     |view      |2025-05-17 15:59:46.960361|p106      |electronics|493.56|
|u4     |view      |2025-

### Zadanie 4 - Bezstanowe zliczanie zdarzeń (źródło plikowe)

In [3]:
print("\n--- URUCHAMIAM ZADANIE 4: Stateless Event Counting ---")
reset_global_batch_counter()

stream_df_task4 = (spark.readStream
             .schema(json_schema)
             .option("maxFilesPerTrigger", 1)
             .json(input_files_path))

# Zliczanie zdarzeń należące do danej grupy event_type
agg1_task4 = stream_df_task4.groupBy("event_type").count()

query_task4 = (agg1_task4
         .writeStream
         .outputMode("complete") 
         .format("console")
         .option("truncate", False)
         .foreachBatch(process_batch_display)
         .queryName("StatelessCountQuery")
         .start())

print(f"Stateless event counting query (Task 4) started. Monitoring: {input_files_path}")
print("Ensure generator.py is running. Processing for a short duration...")
try:
    query_task4.awaitTermination(timeout=30)
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")
    
if query_task4.isActive:
    print("Stopping StatelessCountQuery (Task 4)...")
    query_task4.stop()
    for _ in range(10): 
        if not query_task4.isActive: break
        time.sleep(0.5)
print("Stateless event counting query (Task 4) finished or stopped.")


--- URUCHAMIAM ZADANIE 4: Stateless Event Counting ---
Global batch display counter has been reset.
Stateless event counting query (Task 4) started. Monitoring: data/stream
Ensure generator.py is running. Processing for a short duration...
--- Batch ID: 0 (Global display count: 1) ---
+----------+-----+
|event_type|count|
+----------+-----+
|purchase  |2    |
|view      |8    |
+----------+-----+

--- Batch ID: 1 (Global display count: 2) ---
+----------+-----+
|event_type|count|
+----------+-----+
|cart      |6    |
|purchase  |4    |
|view      |13   |
+----------+-----+

--- Batch ID: 2 (Global display count: 3) ---
+----------+-----+
|event_type|count|
+----------+-----+
|purchase  |5    |
|cart      |7    |
|view      |21   |
+----------+-----+

--- Batch ID: 3 (Global display count: 4) ---
+----------+-----+
|event_type|count|
+----------+-----+
|purchase  |5    |
|cart      |9    |
|view      |25   |
+----------+-----+

--- Batch ID: 4 (Global display count: 5) ---
+----------+

### Zadanie 5.1 - Agregacja w oknach czasowych (Tumbling Window)

In [3]:
print("\n--- URUCHAMIAM ZADANIE 5.1: Windowed Aggregation (Tumbling Window) ---")
reset_global_batch_counter()

stream_df_task5 = (spark.readStream
             .schema(json_schema)
             .option("maxFilesPerTrigger", 1)
             .json(input_files_path))

# Strumień z watermarkiem
stream_with_watermark_task5 = stream_df_task5.withWatermark("timestamp", "1 minute")

# Tumbling window
windowed_tumbling_task5 = (stream_with_watermark_task5
                     .groupBy(
                         window("timestamp", "5 minutes"), # Okno kubłowe
                         "event_type")
                     .count())

query_tumbling_task5 = (
    windowed_tumbling_task5.writeStream
    .outputMode("update") 
    .format("console")
    .option("truncate", False)
    .foreachBatch(process_batch_display)
    .queryName("TumblingWindowQuery")
    .start()
)

print(f"Tumbling window query (Task 5.1) started. Monitoring: {input_files_path}")
print("Ensure generator.py is running. Processing for a short duration (note: output might appear with delay due to windowing and watermark)...")
try:
    query_tumbling_task5.awaitTermination(timeout=45) # Dłuższy czas, aby okna mogły się zamknąć
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_tumbling_task5.isActive:
    print("Stopping TumblingWindowQuery (Task 5.1)...")
    query_tumbling_task5.stop()
    for _ in range(10): 
        if not query_tumbling_task5.isActive: break
        time.sleep(0.5)
print("Tumbling window query (Task 5.1) finished or stopped.")


--- URUCHAMIAM ZADANIE 5.1: Windowed Aggregation (Tumbling Window) ---
Global batch display counter has been reset.
Tumbling window query (Task 5.1) started. Monitoring: data/stream
Ensure generator.py is running. Processing for a short duration (note: output might appear with delay due to windowing and watermark)...
--- Batch ID: 0 (Global display count: 1) ---
+------------------------------------------+----------+-----+
|window                                    |event_type|count|
+------------------------------------------+----------+-----+
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|view      |6    |
|{2025-05-17 15:55:00, 2025-05-17 16:00:00}|view      |2    |
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|purchase  |2    |
+------------------------------------------+----------+-----+

--- Batch ID: 1 (Global display count: 2) ---
+------------------------------------------+----------+-----+
|window                                    |event_type|count|
+----------------------------

### Zadanie 5.2 - Agregacja w oknach czasowych (Sliding Window)

In [4]:
print("\n--- URUCHAMIAM ZADANIE 5.2: Windowed Aggregation (Sliding Window) ---")
reset_global_batch_counter()

# Możemy ponownie użyć definicji stream_df_task5 lub zdefiniować nową, jeśli chcemy izolacji
stream_df_task5_sliding = (spark.readStream # Nowa instancja strumienia dla czystości
             .schema(json_schema)
             .option("maxFilesPerTrigger", 1)
             .json(input_files_path))

stream_with_watermark_task5_sliding = stream_df_task5_sliding.withWatermark("timestamp", "1 minute")

# Sliding window
windowed_sliding_task5 = (stream_with_watermark_task5_sliding
                    .groupBy(
                        window("timestamp", "5 minutes", "1 minute"), # Okno przesuwne
                        "event_type")
                    .count())

query_sliding_task5 = (
    windowed_sliding_task5.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .foreachBatch(process_batch_display)
    .queryName("SlidingWindowQuery")
    .start()
)

print(f"Sliding window query (Task 5.2) started. Monitoring: {input_files_path}")
print("Ensure generator.py is running. Processing for a short duration (note: output might appear with delay)...")
try:
    query_sliding_task5.awaitTermination(timeout=45) # Dłuższy czas
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_sliding_task5.isActive:
    print("Stopping SlidingWindowQuery (Task 5.2)...")
    query_sliding_task5.stop()
    for _ in range(10): 
        if not query_sliding_task5.isActive: break
        time.sleep(0.5)
print("Sliding window query (Task 5.2) finished or stopped.")


--- URUCHAMIAM ZADANIE 5.2: Windowed Aggregation (Sliding Window) ---
Global batch display counter has been reset.
Sliding window query (Task 5.2) started. Monitoring: data/stream
Ensure generator.py is running. Processing for a short duration (note: output might appear with delay)...
--- Batch ID: 0 (Global display count: 1) ---
+------------------------------------------+----------+-----+
|window                                    |event_type|count|
+------------------------------------------+----------+-----+
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|view      |6    |
|{2025-05-17 15:58:00, 2025-05-17 16:03:00}|view      |8    |
|{2025-05-17 15:56:00, 2025-05-17 16:01:00}|view      |5    |
|{2025-05-17 16:02:00, 2025-05-17 16:07:00}|view      |1    |
|{2025-05-17 16:01:00, 2025-05-17 16:06:00}|purchase  |2    |
|{2025-05-17 15:57:00, 2025-05-17 16:02:00}|purchase  |1    |
|{2025-05-17 16:02:00, 2025-05-17 16:07:00}|purchase  |1    |
|{2025-05-17 15:59:00, 2025-05-17 16:04:00}|vie

### Segmentacja Klientów

In [ ]:
print("\n--- URUCHAMIAM ZADANIE GŁÓWNE: Segmentacja Klientów ---")
reset_global_batch_counter()

stream_df_segmentation = (spark.readStream
             .schema(json_schema)
             .option("maxFilesPerTrigger", 2) # Przetwarzaj więcej plików na batch dla tego zadania
             .json(input_files_path))

stream_with_watermark_seg = stream_df_segmentation.withWatermark("timestamp", "1 minute")

user_activities_seg = (stream_with_watermark_seg
                   .groupBy(
                       window("timestamp", "5 minutes"), 
                       col("user_id")
                   )
                   .agg(collect_set("event_type").alias("event_types_collected")))

segmentation_logic = """
    CASE
        WHEN array_contains(event_types_collected, 'purchase') THEN 'Buyer'
        WHEN array_contains(event_types_collected, 'cart') AND NOT array_contains(event_types_collected, 'purchase') THEN 'Cart Abandoner'
        WHEN array_contains(event_types_collected, 'view') AND NOT array_contains(event_types_collected, 'cart') AND NOT array_contains(event_types_collected, 'purchase') THEN 'Lurker'
        ELSE 'Inactive/Other'
    END
"""
# Poprawiona logika segmentacji dla większej precyzji

customer_segments_seg = user_activities_seg.withColumn("segment", expr(segmentation_logic))

query_segmentation = (customer_segments_seg.writeStream
                      .outputMode("update") 
                      .format("console")
                      .option("truncate", False)
                      .foreachBatch(process_batch_display)
                      .queryName("CustomerSegmentationQuery")
                      .start())

print(f"Customer segmentation query started. Monitoring: {input_files_path}")
print("Ensure generator.py is running. This query might need more time to show diverse segments...")
try:
    query_segmentation.awaitTermination(timeout=50) # Dłuższy czas na obserwację segmentacji
except Exception as e:
    print(f"AwaitTermination interrupted or failed: {e}")

if query_segmentation.isActive:
    print("Stopping CustomerSegmentationQuery...")
    query_segmentation.stop()
    for _ in range(10): 
        if not query_segmentation.isActive: break
        time.sleep(0.5)
print("Customer segmentation query finished or stopped.")


--- URUCHAMIAM ZADANIE GŁÓWNE: Segmentacja Klientów ---
Global batch display counter has been reset.
Customer segmentation query started. Monitoring: data/stream
Ensure generator.py is running. This query might need more time to show diverse segments...
--- Batch ID: 0 (Global display count: 1) ---
+------------------------------------------+-------+----------------------+--------------+
|window                                    |user_id|event_types_collected |segment       |
+------------------------------------------+-------+----------------------+--------------+
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|u4     |[cart, view]          |Cart Abandoner|
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|u2     |[cart, view]          |Cart Abandoner|
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|u9     |[view]                |Lurker        |
|{2025-05-17 15:55:00, 2025-05-17 16:00:00}|u9     |[purchase]            |Buyer         |
|{2025-05-17 16:00:00, 2025-05-17 16:05:00}|u6     |[view]    

### Zatrzymanie Sesji Spark

In [3]:
print("Attempting to stop all active streams and the Spark session...")

if 'spark_session_global' in globals() and spark_session_global:
    active_streams = spark_session_global.streams.active
    if active_streams:
        print(f"Found {len(active_streams)} active streams. Stopping them...")
        for s_query in active_streams:
            try:
                print(f"Stopping stream: {s_query.name or s_query.id}")
                s_query.stop()
                # Dodatkowe oczekiwanie na zatrzymanie strumienia
                for _ in range(10): # max 5 sekund czekania
                    if not s_query.isActive: 
                        print(f"Stream {s_query.name or s_query.id} stopped.")
                        break
                    time.sleep(0.5)
                if s_query.isActive:
                     print(f"Stream {s_query.name or s_query.id} did not stop gracefully.")
            except Exception as e:
                print(f"Error stopping stream {s_query.name or s_query.id}: {e}")
    
    print("Stopping SparkSession...")
    spark_session_global.stop()
    spark_session_global = None # Usuń referencję
    print("SparkSession stopped.")
else:
    print("No active SparkSession found or already stopped.")

# Wyczyść listę aktywnych strumieni Sparka, jeśli to konieczne (dla niektórych środowisk Jupyter)
try:
    from pyspark.sql.streaming import StreamingQueryManager
    if spark.streams is not None: # spark może być jeszcze zdefiniowane z create_spark_session
      if hasattr(spark.streams, '_sqm') and isinstance(spark.streams._sqm, StreamingQueryManager): # Spark 3.x
          spark.streams._sqm.active = [] # type: ignore
      elif hasattr(spark.streams, 'active') and isinstance(spark.streams.active, list): # Starsze wersje Sparka
          spark.streams.active = [] # type: ignore
      print("Cleared Spark's internal list of active streams (if applicable).")
except Exception as e:
    print(f"Could not clear Spark's active streams list: {e}")

Attempting to stop all active streams and the Spark session...
Stopping SparkSession...
SparkSession stopped.
Could not clear Spark's active streams list: property 'active' of 'StreamingQueryManager' object has no setter
